# SST1 RSoXS Direct Tiled Dashboard

Interactive dashboard for live and post-run scan inspection without the legacy loader abstraction.

- Connect directly to Tiled (`from_profile` then URI fallback).
- Filter and group scans by sample and run type.
- Select scans and plot arbitrary x/y traces.
- Apply simple NEXAFS transforms (`TEY/I0`, post-edge normalization).
- Enable periodic refresh for near-real-time experiment monitoring.


In [6]:
import os

import numpy as np
import pandas as pd
import panel as pn
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from tiled.client import from_profile, from_uri

pn.extension("tabulator")
os.environ.setdefault("TILED_SITE_PROFILES", "/nsls2/software/etc/tiled/profiles")


'/nsls2/software/etc/tiled/profiles'

In [7]:
class SST1DirectTiledDashboard:
    """Interactive SST1 dashboard backed by direct Tiled access."""

    def __init__(self, tiled_uri="https://tiled.nsls2.bnl.gov", beamline="rsoxs", stream="raw", profile="rsoxs"):
        self.tiled_uri = tiled_uri
        self.beamline = beamline
        self.stream = stream
        self.profile = profile
        self.catalog = None
        self.authenticated = False
        self.catalog_df = pd.DataFrame()
        self.scan_cache = {}
        self._refresh_callback = None

        self.username = pn.widgets.TextInput(name="Username", placeholder="BNL username", width=180)
        self.password = pn.widgets.PasswordInput(name="Password", placeholder="BNL password", width=180)
        self.login_btn = pn.widgets.Button(name="Authenticate", button_type="primary", width=130)

        self.last_n = pn.widgets.IntInput(name="Latest runs", value=300, start=1, width=140)
        self.cycle = pn.widgets.Select(name="Cycle", options=["Any"], value="Any", width=140)
        self.institution = pn.widgets.Select(name="Institution", options=["Any"], value="Any", width=160)
        self.sample_id = pn.widgets.Select(name="Sample", options=["Any"], value="Any", width=220)
        self.run_type = pn.widgets.Select(name="Run type", options=["Any"], value="Any", width=180)
        self.plan_filter = pn.widgets.TextInput(name="Plan regex", placeholder="carbon|oxygen|nexafs", width=220)
        self.refresh_btn = pn.widgets.Button(name="Refresh", button_type="primary")
        self.auto_refresh = pn.widgets.Toggle(name="Auto refresh", button_type="light", value=False, width=120)
        self.refresh_sec = pn.widgets.IntInput(name="Every seconds", value=20, start=5, width=130)

        self.group_by = pn.widgets.Select(name="Group by", options=["none", "sample_id", "run_type", "sample_id + run_type"], value="sample_id + run_type", width=220)
        self.scan_select = pn.widgets.MultiChoice(name="Scans", options=[], value=[], width=700)

        self.x_var = pn.widgets.Select(name="X", options=[], width=200)
        self.y_var = pn.widgets.Select(name="Y", options=[], width=280)
        self.y_transform = pn.widgets.Select(name="Transform", options=["raw", "tey_over_i0", "post_edge_norm"], value="raw", width=180)
        self.post_edge_min = pn.widgets.FloatInput(name="Post-edge min", value=340.0, width=130)
        self.post_edge_max = pn.widgets.FloatInput(name="Post-edge max", value=345.0, width=130)

        self.status = pn.pane.Markdown("Authenticate with username/password.", height=30)
        self.group_summary = pn.pane.Markdown("", height=170)
        self.plot_pane = pn.pane.Bokeh(height=430, sizing_mode="stretch_width")
        self.table = pn.pane.DataFrame(pd.DataFrame(), height=320, sizing_mode="stretch_width")

        self.login_btn.on_click(self._authenticate)
        self.refresh_btn.on_click(self._refresh_catalog)
        self.auto_refresh.param.watch(self._toggle_auto_refresh, "value")
        self.scan_select.param.watch(self._on_scan_selection_change, "value")

        watched = [self.cycle, self.institution, self.sample_id, self.run_type, self.plan_filter, self.group_by, self.x_var, self.y_var, self.y_transform, self.post_edge_min, self.post_edge_max]
        for widget in watched:
            widget.param.watch(self._react, "value")

        self.view = pn.Column(
            pn.Row(self.username, self.password, self.login_btn),
            pn.Row(self.refresh_btn, self.auto_refresh, self.refresh_sec, self.last_n, self.status),
            pn.Row(self.cycle, self.institution, self.sample_id, self.run_type, self.plan_filter),
            pn.Row(self.group_by),
            self.group_summary,
            self.table,
            self.scan_select,
            pn.Row(self.x_var, self.y_var, self.y_transform, self.post_edge_min, self.post_edge_max),
            self.plot_pane,
        )
    def _connect_catalog(self, username, password):
        """Create a direct connection to the SST1 RSoXS Tiled catalog."""
        try:
            return from_profile(self.profile)
        except Exception:
            root = from_uri(self.tiled_uri, username=username, password=password)
            return root[self.beamline][self.stream]

    def _authenticate(self, *_):
        """Authenticate user and initialize catalog access."""
        user = (self.username.value or "").strip()
        pwd = self.password.value or ""
        if user == "" or pwd == "":
            self._set_status("Username and password are required.", error=True)
            return
        try:
            self.catalog = self._connect_catalog(user, pwd)
            self.authenticated = True
            self._set_status("Authentication successful.")
            self._refresh_catalog()
        except Exception as exc:
            self.catalog = None
            self.authenticated = False
            self._set_status(f"Authentication failed: {exc}", error=True)

    def _set_status(self, text, error=False):
        """Update status line text."""
        self.status.object = f"**Error:** {text}" if error else text

    def _safe_start(self, run, key, default="N/A"):
        """Get a value from start document with fallback."""
        return run.start.get(key, default) if hasattr(run, "start") else default

    def _safe_stop_num_events(self, run):
        """Extract primary point count from stop document."""
        try:
            return int(run.stop.get("num_events", {}).get("primary", 0))
        except Exception:
            return 0

    def _infer_run_type(self, plan_name):
        """Infer run type from plan name."""
        s = str(plan_name).lower()
        if any(x in s for x in ["nexafs", "xafs", "carbon", "oxygen", "fluorine", "nitrogen", "energy"]):
            return "energy_scan"
        if any(x in s for x in ["angle", "theta", "th", "rock", "grazing"]):
            return "angle_scan"
        if "count" in s:
            return "count"
        return "other"

    def _catalog_to_df(self, limit):
        """Build a metadata dataframe from latest catalog runs."""
        rows = []
        items = list(self.catalog.items())
        if limit is not None and limit > 0:
            items = items[-limit:]
        for uid, run in items:
            start_time = self._safe_start(run, "time", np.nan)
            try:
                start_time = pd.to_datetime(start_time, unit="s", utc=True).tz_convert(None)
            except Exception:
                start_time = pd.NaT
            plan_name = self._safe_start(run, "plan_name", "N/A")
            rows.append({
                "uid": str(uid),
                "scan_id": str(self._safe_start(run, "scan_id", "N/A")),
                "start_time": start_time,
                "cycle": str(self._safe_start(run, "cycle", "N/A")),
                "institution": str(self._safe_start(run, "institution", "N/A")),
                "project": str(self._safe_start(run, "project_name", "N/A")),
                "sample_name": str(self._safe_start(run, "sample_name", "N/A")),
                "sample_id": str(self._safe_start(run, "sample_id", "N/A")),
                "plan_name": str(plan_name),
                "run_type": self._infer_run_type(plan_name),
                "exit_status": str(getattr(getattr(run, "stop", {}), "get", lambda *_: "N/A")("exit_status", "N/A")),
                "num_points": self._safe_stop_num_events(run),
            })
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.sort_values("start_time", ascending=False, na_position="last").reset_index(drop=True)
        return df

    def _refresh_catalog(self, *_):
        """Refresh metadata table from Tiled."""
        if not self.authenticated or self.catalog is None:
            self._set_status("Authenticate first.", error=True)
            return
        try:
            previous = set(self.scan_select.value or [])
            self.catalog_df = self._catalog_to_df(self.last_n.value)
            if self.catalog_df.empty:
                self._set_table_value(pd.DataFrame())
                self.scan_select.options = []
                self.scan_select.value = []
                self.group_summary.object = "No runs found."
                self.plot_pane.object = self._empty_figure()
                self._set_status("Catalog returned no runs.", error=True)
                return
            self._update_filter_options()
            filtered = self._filtered_catalog()
            show_cols = ["scan_id", "start_time", "sample_id", "sample_name", "run_type", "plan_name", "cycle", "institution", "num_points", "exit_status"]
            self._set_table_value(filtered[show_cols].copy() if not filtered.empty else pd.DataFrame(columns=show_cols))
            scan_options = filtered["scan_id"].astype(str).tolist()
            self.scan_select.options = scan_options
            kept = [sid for sid in previous if sid in scan_options]
            if kept:
                self.scan_select.value = kept
            else:
                self.scan_select.value = scan_options[: min(4, len(scan_options))]
            self._update_group_summary(filtered)
            self._update_axis_options()
            self._set_status(f"Loaded {len(filtered)} run(s) from latest catalog window.")
        except Exception as exc:
            self._set_status(str(exc), error=True)

    def _update_filter_options(self):
        """Populate filter widget choices from current metadata."""
        df = self.catalog_df
        self.cycle.options = ["Any"] + sorted(df["cycle"].dropna().astype(str).unique().tolist())
        self.institution.options = ["Any"] + sorted(df["institution"].dropna().astype(str).unique().tolist())
        self.sample_id.options = ["Any"] + sorted(df["sample_id"].dropna().astype(str).unique().tolist())
        self.run_type.options = ["Any"] + sorted(df["run_type"].dropna().astype(str).unique().tolist())

    def _set_table_value(self, frame):
        """Replace table data for the DataFrame pane."""
        self.table.object = frame

    def _filtered_catalog(self):
        """Apply UI filters to metadata dataframe."""
        df = self.catalog_df.copy()
        if self.cycle.value != "Any":
            df = df[df["cycle"] == self.cycle.value]
        if self.institution.value != "Any":
            df = df[df["institution"] == self.institution.value]
        if self.sample_id.value != "Any":
            df = df[df["sample_id"] == self.sample_id.value]
        if self.run_type.value != "Any":
            df = df[df["run_type"] == self.run_type.value]
        if self.plan_filter.value.strip() != "":
            df = df[df["plan_name"].str.contains(self.plan_filter.value.strip(), case=False, regex=True, na=False)]
        return df.reset_index(drop=True)

    def _update_group_summary(self, filtered):
        """Render grouped run counts."""
        if filtered.empty:
            self.group_summary.object = "No grouped results."
            return
        mode = self.group_by.value
        if mode == "none":
            self.group_summary.object = f"Total runs: **{len(filtered)}**"
            return
        keys = ["sample_id", "run_type"] if mode == "sample_id + run_type" else [mode]
        grouped = filtered.groupby(keys, dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)
        self.group_summary.object = grouped.to_markdown(index=False)

    def _get_run(self, scan_id):
        """Resolve run by scan id from current catalog slice."""
        match = self.catalog_df[self.catalog_df["scan_id"].astype(str) == str(scan_id)]
        if match.empty:
            raise KeyError(f"Scan id {scan_id} not in current catalog window")
        uid = match.iloc[0]["uid"]
        return self.catalog[uid]

    def _load_scan_data(self, scan_id):
        """Load primary xarray dataset for a scan."""
        key = str(scan_id)
        if key in self.scan_cache:
            return self.scan_cache[key]
        run = self._get_run(scan_id)
        ds = run["primary"]["data"].read()
        self.scan_cache[key] = ds
        return ds

    def _candidate_x(self, ds):
        """Suggest x-axis variables from a dataset."""
        variables = set(getattr(ds, "variables", {}).keys())
        dims = list(getattr(ds, "sizes", {}).keys())
        out = []
        for name in ["en_energy", "en_energy_setpoint", "energy", "Energy", "time"]:
            if name in variables or name in dims:
                out.append(name)
        for name in dims:
            if name not in out:
                out.append(name)
        if not out:
            out = list(getattr(ds, "data_vars", {}).keys())
        return out

    def _candidate_y(self, ds):
        """Suggest y-axis variables from data vars."""
        y = list(getattr(ds, "data_vars", {}).keys())
        preferred = ["RSoXS Sample Current", "TEY", "WAXS Beamstop", "DM7 photodiode", "Photodiode", "RSoXS Au Mesh Current", "I0"]
        ordered = [v for v in preferred if v in y]
        for v in y:
            if v not in ordered:
                ordered.append(v)
        return ordered

    def _on_scan_selection_change(self, *_):
        """React to scan selection updates."""
        self._update_axis_options()

    def _update_axis_options(self):
        """Refresh axis options from first selected scan."""
        scans = list(self.scan_select.value or [])
        if not scans:
            self.x_var.options = []
            self.y_var.options = []
            self.plot_pane.object = self._empty_figure()
            return
        try:
            ds = self._load_scan_data(scans[0])
            x_opts = self._candidate_x(ds)
            y_opts = self._candidate_y(ds)
            self.x_var.options = x_opts
            self.y_var.options = y_opts
            if x_opts and self.x_var.value not in x_opts:
                self.x_var.value = x_opts[0]
            if y_opts and self.y_var.value not in y_opts:
                self.y_var.value = y_opts[0]
            self._update_plot()
        except Exception as exc:
            self._set_status(str(exc), error=True)

    def _to_1d(self, arr):
        """Convert an xarray-like variable to a flat numpy array."""
        try:
            return np.asarray(arr.values).reshape(-1)
        except Exception:
            return np.asarray(list(arr)).reshape(-1)

    def _compute_transformed_y(self, ds, x, y_raw):
        """Apply selected y transform for NEXAFS quick-look."""
        y = y_raw.astype(float)
        mode = self.y_transform.value
        if mode == "tey_over_i0":
            i0_name = "I0" if "I0" in ds.data_vars else "RSoXS Au Mesh Current"
            if i0_name not in ds.data_vars:
                return np.full_like(y, np.nan, dtype=float)
            i0 = self._to_1d(ds[i0_name])[: len(y)].astype(float)
            with np.errstate(divide="ignore", invalid="ignore"):
                y = np.where(np.abs(i0) > 0, y / i0, np.nan)
        if mode == "post_edge_norm":
            lo = float(self.post_edge_min.value)
            hi = float(self.post_edge_max.value)
            mask = (x >= lo) & (x <= hi)
            baseline = np.nanmean(y[mask]) if np.any(mask) else np.nan
            if np.isfinite(baseline) and baseline != 0:
                y = y / baseline
            else:
                y = np.full_like(y, np.nan, dtype=float)
        return y

    def _single_scan_df(self, scan_id):
        """Create plotting dataframe for one scan."""
        ds = self._load_scan_data(scan_id)
        x_name = self.x_var.value
        y_name = self.y_var.value
        if x_name not in ds and x_name not in ds.sizes:
            return pd.DataFrame(columns=["x", "y", "scan_id"])
        if y_name not in ds.data_vars:
            return pd.DataFrame(columns=["x", "y", "scan_id"])
        x = self._to_1d(ds[x_name])
        y_raw = self._to_1d(ds[y_name])
        n = min(len(x), len(y_raw))
        x = x[:n]
        y_raw = y_raw[:n]
        y = self._compute_transformed_y(ds, x, y_raw)
        frame = pd.DataFrame({"x": x, "y": y, "scan_id": str(scan_id)})
        frame = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=["x", "y"])
        return frame

    def _empty_figure(self):
        """Create an empty Bokeh figure with axes."""
        p = figure(height=400, sizing_mode="stretch_width", title="No data selected", tools="pan,wheel_zoom,box_zoom,reset,save")
        p.xaxis.axis_label = "x"
        p.yaxis.axis_label = "y"
        return p

    def _transform_label(self):
        """Label for selected transform mode."""
        mapping = {"raw": "raw", "tey_over_i0": "TEY/I0", "post_edge_norm": "post-edge normalized"}
        return mapping.get(self.y_transform.value, self.y_transform.value)

    def _update_plot(self):
        """Render interactive Bokeh overlay for selected scans."""
        scans = list(self.scan_select.value or [])
        if not scans or not self.x_var.value or not self.y_var.value:
            self.plot_pane.object = self._empty_figure()
            return
        rows = []
        errors = []
        for sid in scans:
            try:
                rows.append(self._single_scan_df(sid))
            except Exception as exc:
                errors.append(f"{sid}: {exc}")
        if not rows:
            self.plot_pane.object = self._empty_figure()
            if errors:
                self._set_status("; ".join(errors), error=True)
            return
        plot_df = pd.concat(rows, ignore_index=True)
        p = figure(height=420, sizing_mode="stretch_width", title=f"{self.y_var.value} vs {self.x_var.value} ({self._transform_label()})", tools="pan,wheel_zoom,box_zoom,reset,save")
        p.xaxis.axis_label = self.x_var.value
        p.yaxis.axis_label = f"{self.y_var.value} [{self._transform_label()}]"
        palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
        for i, sid in enumerate(scans):
            sdf = plot_df[plot_df["scan_id"] == str(sid)]
            if sdf.empty:
                continue
            src = ColumnDataSource(sdf)
            renderer = p.line("x", "y", source=src, line_width=2, color=palette[i % len(palette)], legend_label=f"scan {sid}")
            p.add_tools(HoverTool(renderers=[renderer], tooltips=[("scan", "@scan_id"), ("x", "@x{0.000}"), ("y", "@y{0.000}")]))
        p.legend.click_policy = "hide"
        self.plot_pane.object = p
        if errors:
            self._set_status("Plotted with skipped scans: " + "; ".join(errors), error=True)
        else:
            self._set_status(f"Plotted {len(scans)} scan(s).")

    def _toggle_auto_refresh(self, event):
        """Start or stop periodic metadata refresh callback."""
        if event.new:
            if not self.authenticated:
                self.auto_refresh.value = False
                self._set_status("Authenticate first.", error=True)
                return
            period_ms = int(max(5, self.refresh_sec.value) * 1000)
            if self._refresh_callback is not None:
                self._refresh_callback.stop()
            self._refresh_callback = pn.state.add_periodic_callback(self._refresh_catalog, period=period_ms, start=True)
            self._set_status(f"Auto refresh enabled every {period_ms // 1000}s.")
        else:
            if self._refresh_callback is not None:
                self._refresh_callback.stop()
                self._refresh_callback = None
            self._set_status("Auto refresh disabled.")

    def _react(self, *_):
        """General reactive handler for filter and plot controls."""
        if not self.authenticated:
            return
        filtered = self._filtered_catalog()
        show_cols = ["scan_id", "start_time", "sample_id", "sample_name", "run_type", "plan_name", "cycle", "institution", "num_points", "exit_status"]
        self._set_table_value(filtered[show_cols].copy() if not filtered.empty else pd.DataFrame(columns=show_cols))
        self._update_group_summary(filtered)
        options = filtered["scan_id"].astype(str).tolist() if not filtered.empty else []
        current = [sid for sid in (self.scan_select.value or []) if sid in options]
        self.scan_select.options = options
        if current:
            self.scan_select.value = current
        elif options:
            self.scan_select.value = options[: min(4, len(options))]
        else:
            self.scan_select.value = []
        self._update_plot()

    def show(self):
        """Return panel layout."""
        return self.view


In [8]:
dashboard = SST1DirectTiledDashboard()
dashboard.show()

BokehModel(combine_events=True, render_bundle={'docs_json': {'60c27747-3d15-4729-baf8-95bbdf631439': {'version…